# Karten

In [99]:
from copy import deepcopy
from dataclasses import dataclass
import json
from pathlib import Path
import uuid
from enum import Enum
from dataclasses import field
import random

UUID_LENGTH = 8
def random_hex(length: int) -> str:
    return ''.join(random.choice("0123456789abcdef") for _ in range(length))

class CardType(str, Enum):
    CREATURE = "creature"
    ENERGY = "energy"
    EVENT = "event"

class EmotionType(str, Enum):
    ANGRY = "wuetend"
    HAPPY = "gluecklich"
    BORED = "gelangweilt"
    SURPRISED = "ueberrascht"
    SCARED = "aengstlich"
    DISGUSTED = "angeekelt"
    SAD = "traurig"
    ROBOT_HAPPY = "neutral_gluecklich"
    ROBOT_UNHAPPY = "neutral_ungluecklich"


@dataclass
class Card:
    name: str
    instance_id: str = field(default_factory=lambda: random_hex(UUID_LENGTH))

    def __str__(self) -> str:
        return f"{self.name} [{self.instance_id}]"

    @property
    def card_type(self) -> CardType:
        raise NotImplementedError

@dataclass(frozen=True)
class Attack:
    name: str
    damage: int
    type: EmotionType
    energy_cost: int
    advantage: int = 0

    @classmethod
    def from_dict(cls, data: dict) -> "Attack":
        return cls(
            name=data["name"],
            damage=int(data["damage"]),
            type=EmotionType(data["type"]),
            advantage=int(data.get("advantage", 0)),
            energy_cost=int(data.get("energy", 0))
        )

    def __str__(self) -> str:
        return f"{self.name} A{self.damage} E{self.energy_cost} +{self.advantage}"

@dataclass
class CreatureCard(Card):
    index: int = -1
    stage: int = 1
    defense: int = 0
    attacks: list[Attack] = field(default_factory=list)

    # evolves_from_index: int | None = None # TODO

    @property
    def card_type(self) -> CardType:
        return CardType.CREATURE

    @classmethod
    def from_dict(cls, data: dict) -> "CreatureCard":
        return cls(
            name=data["name"],
            index=int(data.get("index", -1)),
            stage=int(data.get("stage", 1)),
            defense=int(data.get("defense", 0)),
            attacks=[Attack.from_dict(a) for a in data.get("attacks", [])]
        )

class CardsPool:
    @staticmethod
    def create_random_deck(size: int) -> list[CreatureCard]:
        base_dir = Path.cwd()
        json_path = base_dir / "cards" / "cards.json"
    
        if not json_path.exists():
            json_path = base_dir.parent / "cards" / "cards.json"

        with json_path.open("r", encoding="utf-8") as f:
            raw_cards = json.load(f)

        deck = []
        for _ in range(size):
            chosen = random.choice(raw_cards)
            deck.append(CreatureCard.from_dict(chosen))
            
        return deck


# Kreaturen

In [100]:
@dataclass
class Creature:
    card_template: CreatureCard
    instance_id: str = field(default_factory=lambda: random_hex(UUID_LENGTH))
    defense: int = 0
    attacks: list[Attack] = field(default_factory=list)

    @classmethod
    def from_card(cls, card: CreatureCard) -> "Creature":
        return cls(card_template=card, defense=card.defense, attacks=card.attacks)

    def __str__(self) -> str:
        return f"{self.card_template.name} [{self.instance_id}] D{self.card_template.defense}: {', '.join(str(attack) for attack in self.card_template.attacks) or 'keine'}"

# Game Logik

In [101]:

class Player:
    def __init__(self, player_name: str, deck: list[Card]) -> None:
        self.display_name = player_name
        self.deck = deck
        self.hand = self.draw(3)
        self.field = []

    def draw(self, count: int = 1) -> list[Card]:
            drawn: list[Card] = []
            for _ in range(count):
                if not self.deck: # if deck is empty
                    break
                drawn.append(self.deck.pop())
            return drawn


class Game:
    def __init__(self, player_A: Player, player_B: Player) -> None:
        self.game_id = random_hex(UUID_LENGTH)
        self.player_A = player_A
        self.player_B = player_B
        self.current_player = self.player_A
        self.opponent_player = self.player_B
        self.turn_number = 1

    def play_creature(self, card_id: str) -> None:
        card = next((c for c in self.current_player.hand if c.instance_id == card_id), None)
        if card:
            self.current_player.hand.remove(card)
            self.current_player.field.append(Creature.from_card(card))

    def attack(self, attacker_id: str, attack_index: int, defender_id: str) -> None:
        attacker = next((c for c in self.current_player.field if c.instance_id == attacker_id), None)
        defender = next((c for c in self.opponent_player.field if c.instance_id == defender_id), None)
        if attacker and defender:
            # Check if the attack is strong enough to defeat the defender
            if attacker.attacks[attack_index].damage >= defender.defense:
                self.opponent_player.field.remove(defender)
            # the defender strikes back
            # use always the first (fast) attack for defense
            if defender.attacks[0].damage >= attacker.defense:
                self.current_player.field.remove(attacker)

    def end_turn(self) -> None:
        self.turn_number += 1
        self.current_player = self.player_A if self.current_player == self.player_B else self.player_B
        self.opponent_player = self.player_B if self.current_player == self.player_A else self.player_A

    def __str__(self) -> str:
        return (
            f"--- {self.turn_number} ---\n"
            f"Hand Player A: {', '.join(str(card) for card in self.player_A.hand) or 'leer'}\n"
            f"Hand Player B: {', '.join(str(card) for card in self.player_B.hand) or 'leer'}\n"
            f"Field Player A: {', '.join(str(creature) for creature in self.player_A.field) or 'leer'}\n"
            f"Field Player B: {', '.join(str(creature) for creature in self.player_B.field) or 'leer'}\n"
        )

# Testrun

In [105]:
random.seed(42)

game = Game(
    player_A=Player("Human", CardsPool.create_random_deck(10)),
    player_B=Player("Bot", CardsPool.create_random_deck(10))
)
print(game)

game.play_creature('826a6fce')
print(game)

game.end_turn()

game.play_creature('77021721')
print(game)

game.attack('72e31027', 0, '66e4d58e')
print(game)

--- 1 ---
Hand Player A: Robo [8478dcb7], Schleggach [826a6fce], Samtange [c87a171a]
Hand Player B: Samtange [d633dbdd], Spinfli [278f64f7], Kruggler [77021721]
Field Player A: leer
Field Player B: leer

--- 1 ---
Hand Player A: Robo [8478dcb7], Samtange [c87a171a]
Hand Player B: Samtange [d633dbdd], Spinfli [278f64f7], Kruggler [77021721]
Field Player A: Schleggach [66e4d58e] D1: Schlecker A1 E0 +1, Lechzer A2 E1 +1
Field Player B: leer

--- 2 ---
Hand Player A: Robo [8478dcb7], Samtange [c87a171a]
Hand Player B: Samtange [d633dbdd], Spinfli [278f64f7]
Field Player A: Schleggach [66e4d58e] D1: Schlecker A1 E0 +1, Lechzer A2 E1 +1
Field Player B: Kruggler [72e31027] D2: Grummler A1 E0 +1, Stampfer A2 E2 +1

--- 2 ---
Hand Player A: Robo [8478dcb7], Samtange [c87a171a]
Hand Player B: Samtange [d633dbdd], Spinfli [278f64f7]
Field Player A: leer
Field Player B: Kruggler [72e31027] D2: Grummler A1 E0 +1, Stampfer A2 E2 +1



# Webserver

In [110]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Emotionskreaturen Singleplayer API")

class NewGameRequest(BaseModel):
    player_name: str
class NewGameResponse(BaseModel):
    game_id: str

@app.post("/new_game", response_model=NewGameResponse)
def new_game(payload: NewGameRequest):
    player_name = payload.player_name.strip()
    if not player_name:
        raise HTTPException(status_code=400, detail="player_name is required")

    game = Game(
        player_A=Player(player_name, CardsPool.create_random_deck(10)),
        player_B=Player("Bot", CardsPool.create_random_deck(10)),
    )

    return {"game_id": game.game_id}

uvicorn.run(app, host="127.0.0.1", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop